# 1 – Imports and setup

In [1]:
import sys
sys.path.append('..')  

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from surprise import Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate

from src.data_loader import load_movielens, load_movielens_df
from src.recommender import create_svd_model, create_knn_user_model, create_knn_item_model, create_random_model
from src.evaluator import evaluate_cross_validation, compare_models

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Setup complete!")

ModuleNotFoundError: No module named 'surprise'

# 2 – Load data

In [ ]:
data = load_movielens()
df = load_movielens_df()
print("Data loaded successfully!")
print(f"Shape: {df.shape}")
print(df.head())

# 3 – Dataset statistics

In [ ]:
print("="*50)
print("DATASET STATISTICS")
print("="*50)
print(f"Number of users: {df['user'].nunique()}")
print(f"Number of items (movies): {df['item'].nunique()}")
print(f"Total ratings: {len(df)}")
print(f"Average rating: {df['rating'].mean():.2f}")
print(f"Rating range: {df['rating'].min()} to {df['rating'].max()}")

#  4 – Rating distribution plot

In [ ]:
plt.figure(figsize=(10,5))
df['rating'].hist(bins=5, edgecolor='black', alpha=0.7)
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.grid(False)
plt.savefig('../results/plots/rating_distribution.png')
plt.show()

# 5 – Train-test split

In [ ]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print(f"Train size: {len(trainset.all_ratings())}")
print(f"Test size: {len(testset)}")

# 6 – Train SVD model

In [ ]:
svd_model = create_svd_model(n_factors=50, n_epochs=20)
svd_model.fit(trainset)
pred_svd = svd_model.test(testset)
rmse_svd = accuracy.rmse(pred_svd)
mae_svd = accuracy.mae(pred_svd)
print(f"SVD - RMSE: {rmse_svd:.4f}, MAE: {mae_svd:.4f}")

# 7 – Train KNN User-based

In [ ]:
knn_user = create_knn_user_model(k=40)
knn_user.fit(trainset)
pred_knn_user = knn_user.test(testset)
rmse_knn_user = accuracy.rmse(pred_knn_user)
print(f"KNN User-based - RMSE: {rmse_knn_user:.4f}")

# 8 – Train KNN Item-based

In [ ]:
knn_item = create_knn_item_model(k=40)
knn_item.fit(trainset)
pred_knn_item = knn_item.test(testset)
rmse_knn_item = accuracy.rmse(pred_knn_item)
print(f"KNN Item-based - RMSE: {rmse_knn_item:.4f}")

# 9 – Train Random (baseline)

In [ ]:
random_model = create_random_model()
random_model.fit(trainset)
pred_random = random_model.test(testset)
rmse_random = accuracy.rmse(pred_random)
print(f"Random (baseline) - RMSE: {rmse_random:.4f}")

# 10 – Compare all models

In [ ]:
models = {
    'SVD': svd_model,
    'KNN-User': knn_user,
    'KNN-Item': knn_item,
    'Random': random_model
}

print("\n=== Model Comparison on Test Set ===")
results = {}
for name, model in models.items():
    pred = model.test(testset)
    rmse = accuracy.rmse(pred, verbose=False)
    mae = accuracy.mae(pred, verbose=False)
    results[name] = {'RMSE': rmse, 'MAE': mae}
    print(f"{name:12} -> RMSE: {rmse:.4f}, MAE: {mae:.4f}")

# 11 – Bar plot of RMSE

In [ ]:
names = list(results.keys())
rmse_values = [results[n]['RMSE'] for n in names]

plt.figure(figsize=(10,6))
bars = plt.bar(names, rmse_values, color=['blue', 'green', 'orange', 'red'])
plt.title('Model Comparison - RMSE (lower is better)')
plt.xlabel('Model')
plt.ylabel('RMSE')
plt.ylim(0, 1.8)
for bar, val in zip(bars, rmse_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', va='bottom')
plt.savefig('../results/plots/model_comparison.png')
plt.show()

# 12 – Cross-validation for SVD

In [ ]:
print("\n=== Cross-validation for SVD (5-fold) ===")
cv_results = cross_validate(svd_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"\nAverage RMSE: {np.mean(cv_results['test_rmse']):.4f}")
print(f"Average MAE:  {np.mean(cv_results['test_mae']):.4f}")

# 13 – Get top-N recommendations for a specific user

In [ ]:
def get_top_n_recommendations(model, user_id, df, n=10):
    all_items = df['item'].unique()
    rated_items = df[df['user'] == user_id]['item'].values
    unrated = [item for item in all_items if item not in rated_items]
    
    preds = [(item, model.predict(user_id, item).est) for item in unrated[:200]]
    preds.sort(key=lambda x: x[1], reverse=True)
    return preds[:n]

# Example: user 1
recs = get_top_n_recommendations(svd_model, 1, df, n=10)
print("Top 10 recommendations for User 1:")
for idx, (item, score) in enumerate(recs, 1):
    print(f"{idx:2}. Movie {item} -> predicted rating: {score:.2f}")

# 14 – Final summary

In [ ]:
print("\n" + "="*50)
print("PROJECT SUMMARY")
print("="*50)
print(f"Dataset: MovieLens 100k")
print(f"Total ratings: {len(df)}")
print(f"Users: {df['user'].nunique()}")
print(f"Movies: {df['item'].nunique()}")
print(f"\nBest performing model: SVD")
print(f"Best RMSE: {min([results[n]['RMSE'] for n in results]):.4f}")
print("\n✅ Project complete! All code executed successfully.")